# Exercise 1: Gene Expression Classification with Feedforward Neural Networks

**Lecture 10: Neural Networks and Deep Learning**  
**Center for Neuroscience and Cell Biology, University of Coimbra**

---

## Learning Objectives

By the end of this notebook, you will be able to:
1. Load and preprocess single-cell RNA-seq data
2. Build a feedforward neural network (FFN) in PyTorch
3. Train the model with proper validation
4. Evaluate classification performance
5. Visualize results and interpret model predictions

## Biological Context

**Problem**: Classify cell types from single-cell RNA-seq gene expression profiles

Single-cell RNA sequencing (scRNA-seq) measures gene expression in individual cells, revealing cellular heterogeneity. Machine learning can automatically classify cells into types (T cells, B cells, monocytes, etc.) based on their expression patterns.

In [ ]:
# Install required packages (run once)
!pip install -q torch torchvision scikit-learn pandas numpy matplotlib seaborn umap-learn

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import umap

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Part 1: Data Generation and Exploration

We'll simulate single-cell RNA-seq data with:
- **2000 genes** (features)
- **5000 cells** (samples)
- **8 cell types** (classes): T cells, B cells, NK cells, Monocytes, Dendritic, Neutrophils, Erythrocytes, Platelets

In real applications, you would load data from 10X Genomics, Seurat objects, or AnnData files.

In [ ]:
# Generate synthetic gene expression data
n_samples = 5000
n_genes = 2000
n_classes = 8

# Create synthetic data with realistic properties
X, y = make_classification(
    n_samples=n_samples,
    n_features=n_genes,
    n_informative=200,  # Marker genes
    n_redundant=100,
    n_classes=n_classes,
    n_clusters_per_class=1,
    class_sep=1.5,
    random_state=42
)

# Add noise and make non-negative (like gene expression counts)
X = np.abs(X) + np.random.exponential(0.5, X.shape)

# Cell type labels
cell_types = ['T cells', 'B cells', 'NK cells', 'Monocytes', 
              'Dendritic', 'Neutrophils', 'Erythrocytes', 'Platelets']

print(f"Dataset shape: {X.shape}")
print(f"Number of cell types: {n_classes}")
print(f"Class distribution:")
unique, counts = np.unique(y, return_counts=True)
for cell_type, count in zip(cell_types, counts):
    print(f"  {cell_type}: {count} cells")

In [ ]:
# Visualize data distribution using UMAP
print("Computing UMAP embedding (this may take a minute)...")
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
embedding = reducer.fit_transform(X)

# Plot UMAP
plt.figure(figsize=(12, 8))
scatter = plt.scatter(embedding[:, 0], embedding[:, 1], 
                      c=y, cmap='tab10', s=1, alpha=0.5)
plt.colorbar(scatter, label='Cell Type', ticks=range(n_classes))
plt.title('UMAP Projection of Gene Expression Data', fontsize=14)
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')

# Add legend
for i, cell_type in enumerate(cell_types):
    plt.scatter([], [], c=plt.cm.tab10(i/10), label=cell_type, s=50)
plt.legend(bbox_to_anchor=(1.15, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

## Part 2: Data Preprocessing

Critical steps for neural networks:
1. **Train/validation/test split**: Prevent data leakage
2. **Normalization**: Scale features to similar ranges
3. **Create PyTorch datasets**: Efficient data loading

In [ ]:
# Split data: 70% train, 15% validation, 15% test
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Training set: {X_train.shape[0]} cells")
print(f"Validation set: {X_val.shape[0]} cells")
print(f"Test set: {X_test.shape[0]} cells")

# Normalize features (important!)
# Fit scaler on training data only to prevent data leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("\nData normalized using StandardScaler")

In [ ]:
# Create PyTorch Dataset
class GeneExpressionDataset(Dataset):
    """Dataset for gene expression data"""
    
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Create datasets
train_dataset = GeneExpressionDataset(X_train_scaled, y_train)
val_dataset = GeneExpressionDataset(X_val_scaled, y_val)
test_dataset = GeneExpressionDataset(X_test_scaled, y_test)

# Create data loaders
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Created data loaders with batch size {batch_size}")
print(f"Number of batches - Train: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}")

## Part 3: Build the Feedforward Neural Network

Architecture:
- **Input layer**: 2000 genes
- **Hidden layer 1**: 512 neurons + ReLU + BatchNorm + Dropout(0.3)
- **Hidden layer 2**: 256 neurons + ReLU + BatchNorm + Dropout(0.3)
- **Hidden layer 3**: 128 neurons + ReLU + BatchNorm + Dropout(0.2)
- **Output layer**: 8 cell types (softmax applied in loss function)

In [ ]:
class GeneExpressionClassifier(nn.Module):
    """
    Feedforward neural network for cell type classification
    """
    
    def __init__(self, n_genes=2000, n_cell_types=8):
        super().__init__()
        
        self.network = nn.Sequential(
            # Layer 1: 2000 -> 512
            nn.Linear(n_genes, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            # Layer 2: 512 -> 256
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            # Layer 3: 256 -> 128
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            # Output layer: 128 -> 8
            nn.Linear(128, n_cell_types)
        )
    
    def forward(self, x):
        return self.network(x)

# Initialize model
model = GeneExpressionClassifier(n_genes=n_genes, n_cell_types=n_classes)
model = model.to(device)

# Print model architecture
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

## Part 4: Training Setup

Components:
- **Loss function**: CrossEntropyLoss (combines softmax + negative log-likelihood)
- **Optimizer**: Adam with learning rate 0.001
- **Learning rate scheduler**: ReduceLROnPlateau (adaptive)
- **Early stopping**: Stop if validation loss doesn't improve

In [ ]:
# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=5, factor=0.5, verbose=True
)

print("Training setup complete!")

In [ ]:
# Training and evaluation functions
def train_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_x, batch_y in loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Statistics
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += batch_y.size(0)
        correct += predicted.eq(batch_y).sum().item()
    
    return total_loss / len(loader), 100.0 * correct / total

def evaluate(model, loader, criterion, device):
    """Evaluate model"""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += batch_y.size(0)
            correct += predicted.eq(batch_y).sum().item()
    
    return total_loss / len(loader), 100.0 * correct / total

## Part 5: Train the Model

We'll train for up to 100 epochs with early stopping (patience=10).

In [ ]:
# Training loop with early stopping
num_epochs = 100
patience = 10
best_val_loss = float('inf')
patience_counter = 0

history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

print("Starting training...")
print("-" * 80)

for epoch in range(num_epochs):
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    
    # Validate
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    # Learning rate scheduling
    scheduler.step(val_loss)
    
    # Print progress
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:3d}/{num_epochs} | "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:5.2f}% | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:5.2f}%")
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), '/tmp/best_model.pt')
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

print("-" * 80)
print("Training complete!")

# Load best model
model.load_state_dict(torch.load('/tmp/best_model.pt'))
print(f"Best validation loss: {best_val_loss:.4f}")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training and Validation Loss', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curves
axes[1].plot(history['train_acc'], label='Train Accuracy', linewidth=2)
axes[1].plot(history['val_acc'], label='Validation Accuracy', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].set_title('Training and Validation Accuracy', fontsize=14)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Part 6: Evaluate on Test Set

Now we evaluate the trained model on the held-out test set.

In [ ]:
# Get predictions on test set
model.eval()
all_predictions = []
all_true = []
all_probs = []

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(device)
        outputs = model(batch_x)
        probs = torch.softmax(outputs, dim=1)
        _, predicted = outputs.max(1)
        
        all_predictions.extend(predicted.cpu().numpy())
        all_true.extend(batch_y.numpy())
        all_probs.extend(probs.cpu().numpy())

all_predictions = np.array(all_predictions)
all_true = np.array(all_true)
all_probs = np.array(all_probs)

# Calculate accuracy
test_accuracy = accuracy_score(all_true, all_predictions)
print(f"Test Set Accuracy: {test_accuracy * 100:.2f}%")
print("\n" + "="*80)
print("Classification Report:")
print("="*80)
print(classification_report(all_true, all_predictions, target_names=cell_types))

In [ ]:
# Confusion matrix
cm = confusion_matrix(all_true, all_predictions)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(12, 10))
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=cell_types, yticklabels=cell_types,
            cbar_kws={'label': 'Normalized Frequency'})
plt.xlabel('Predicted Cell Type', fontsize=12)
plt.ylabel('True Cell Type', fontsize=12)
plt.title('Confusion Matrix (Normalized)', fontsize=14)
plt.tight_layout()
plt.show()

## Part 7: Interpretation and Analysis

Let's analyze which genes the model finds most important for classification.

In [ ]:
# Get first layer weights (input -> hidden1)
first_layer_weights = model.network[0].weight.data.cpu().numpy()  # Shape: (512, 2000)

# Calculate importance score for each gene (sum of absolute weights)
gene_importance = np.abs(first_layer_weights).sum(axis=0)

# Get top 20 important genes
top_genes_idx = np.argsort(gene_importance)[-20:]
top_genes_importance = gene_importance[top_genes_idx]

# Plot
plt.figure(figsize=(10, 8))
plt.barh(range(20), top_genes_importance, color='steelblue')
plt.yticks(range(20), [f"Gene {i}" for i in top_genes_idx])
plt.xlabel('Importance Score', fontsize=12)
plt.ylabel('Gene', fontsize=12)
plt.title('Top 20 Most Important Genes for Classification', fontsize=14)
plt.tight_layout()
plt.show()

print("Note: In real data, these would be actual gene names (CD3D, CD4, etc.)")

## Part 8: Prediction on New Cells

Demonstrate how to use the trained model for inference.

In [ ]:
def predict_cell_type(model, gene_expression, scaler, device, cell_types):
    """
    Predict cell type for a single cell's gene expression profile
    
    Parameters:
    -----------
    model : nn.Module
        Trained model
    gene_expression : ndarray
        Gene expression values (shape: [n_genes])
    scaler : StandardScaler
        Fitted scaler from training
    device : torch.device
        Device to run inference on
    cell_types : list
        List of cell type names
    
    Returns:
    --------
    predicted_type : str
        Predicted cell type
    confidence : float
        Prediction confidence (0-1)
    all_probabilities : dict
        Probabilities for all cell types
    """
    model.eval()
    
    # Preprocess
    gene_expression_scaled = scaler.transform(gene_expression.reshape(1, -1))
    gene_expression_tensor = torch.FloatTensor(gene_expression_scaled).to(device)
    
    # Predict
    with torch.no_grad():
        output = model(gene_expression_tensor)
        probabilities = torch.softmax(output, dim=1)[0].cpu().numpy()
    
    # Get prediction
    predicted_idx = probabilities.argmax()
    predicted_type = cell_types[predicted_idx]
    confidence = probabilities[predicted_idx]
    
    # All probabilities
    all_probs = {cell_type: prob for cell_type, prob in zip(cell_types, probabilities)}
    
    return predicted_type, confidence, all_probs

# Example prediction on a test sample
sample_idx = 0
sample_expression = X_test[sample_idx]
true_label = y_test[sample_idx]

pred_type, conf, all_probs = predict_cell_type(
    model, sample_expression, scaler, device, cell_types
)

print("="*80)
print("Single Cell Prediction Example")
print("="*80)
print(f"True cell type: {cell_types[true_label]}")
print(f"Predicted cell type: {pred_type}")
print(f"Confidence: {conf:.2%}")
print("\nAll probabilities:")
for cell_type, prob in sorted(all_probs.items(), key=lambda x: x[1], reverse=True):
    print(f"  {cell_type:15s}: {prob:.2%}")

## Summary and Key Takeaways

### What We Learned

1. **Data preprocessing is critical**:
   - Proper train/val/test splitting
   - Feature normalization (StandardScaler)
   - Preventing data leakage

2. **FFN architecture choices**:
   - Multiple hidden layers with decreasing sizes
   - BatchNormalization for training stability
   - Dropout for regularization
   - ReLU activation functions

3. **Training best practices**:
   - Learning rate scheduling
   - Early stopping to prevent overfitting
   - Monitoring validation metrics
   - Saving best model checkpoints

4. **Evaluation**:
   - Multiple metrics (accuracy, precision, recall, F1)
   - Confusion matrix analysis
   - Per-class performance

### Extensions to Try

1. **Hyperparameter tuning**:
   - Try different network architectures (wider, deeper)
   - Experiment with dropout rates
   - Test different optimizers (SGD, AdamW)

2. **Data augmentation**:
   - Add Gaussian noise
   - Feature dropout

3. **Class imbalance**:
   - Weighted loss function
   - Oversampling rare cell types

4. **Real data**:
   - Load actual scRNA-seq data (10X, Seurat)
   - Use established marker genes
   - Compare with traditional methods (Seurat, Scanpy)

### Next Steps

- **Exercise 2**: Cell image segmentation with CNNs
- **Exercise 3**: Protein sequence analysis with RNNs
- **Exercise 4**: Transfer learning with protein language models
- **Exercise 5**: AlphaFold protein structure prediction